# Interactive cluster size distribution visualisation

<picture>
  <!-- Shown when the system/browser is in light mode -->
  <source srcset="notebook_resources/mgnify_wordmark_dark_on_light.svg" media="(prefers-color-scheme: light)">
  
  <!-- Fallback/Default shown in dark mode -->
  <img src="notebook_resources/mgnify_wordmark_light_on_dark.svg" width="300" alt="MGnify logo">
</picture>

This notebook serves as a tool to analyse the cluster size distribution, helping you set or adjust the `min_cluster_size` for the main pipeline or for `separate_clusters.py`.

Before running this notebook, ensure you have the required dependencies installed and the cluster `.tsv` file ready for analysis.

## Import libraries

In [ ]:
# %pip install ipyfilechooser pandas matplotlib numpy ipywidgets

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ipyfilechooser import FileChooser
from IPython.display import HTML, clear_output, display

## Create the filechooser widget

Select the `cluster.tsv` file generated by the MiSSN pipeline or the `linclust.sh` script. It can be found under the `/linclust_results` subdirectory.

> **Note:** If you are using a custom file that was **not** generated by the MiSSN pipeline, please ensure it meets the required format: a `.tsv` file with no headers, where the **Cluster ID** is located in the first column (Column 0).

In [ ]:
# '..' means it will start looking in the root directory of MiSSN
fc = FileChooser('..')
fc.title = '<b>Select your cluster.tsv file</b>'
fc.filter_pattern = '*.tsv'  # Only show results with .tsv extension 

load_button = widgets.Button(description="Load data", button_style='primary')
output = widgets.Output()

# Define what happens when the button is clicked
def load_data(b):
    """
    Callback function to read the selected TSV file, count the members 
    per cluster, and load the aggregated data into global variables.
    """
    with output:
        clear_output()
        
        if not fc.selected:
            print("Please select a file from the picker above first!")
            return
            
        FILE_PATH = fc.selected
        CLUSTER_COL = 0
        
        print(f"Loading data from {FILE_PATH}...")
        
        # Use 'global' so df and all_cluster_sizes are saved into the notebook's 
        # memory and can be used in other cells later
        global df, all_cluster_sizes 
        
        df = pd.read_csv(FILE_PATH, sep="\t", header=None)
        edges_df = df  # Assigning df to edges_df so it can be previewed
        all_cluster_sizes = df.groupby(CLUSTER_COL).size()

        print("Data loaded successfully!")
        print(f"Total number of clusters: {all_cluster_sizes.size}")
        print(f"Max cluster size: {all_cluster_sizes.max()}")
        
        display(edges_df.head(5))

# Attach the function to the button
load_button.on_click(load_data)

display(fc, load_button, output)

## Define the stats helper function

This cell handles HTML string generation, separating it from the plotting logic.

In [ ]:
def display_cluster_stats(cluster_sizes, min_size):
    """Generates and displays the HTML statistics banner."""
    if cluster_sizes.size == 0:
        display(HTML(f"<div style='color: #FF5252; padding: 10px;'><b>No clusters found with &ge; {min_size} members. Try a lower minimum.</b></div>"))
        return False # Return False to indicate no data

    stats_html = f"""
    <div style="display: flex; gap: 20px; background-color: #f4f6f8; padding: 15px 20px; border-radius: 8px; border: 1px solid #e1e4e8; font-family: sans-serif; margin-bottom: 15px;">
        
        <div style="width: 180px;">
             <span style="color: #586069; font-size: 0.9em; text-transform: uppercase;">Clusters (&ge; {min_size})</span><br>
             <span style="font-size: 1.4em; font-weight: bold; color: #24292e;">{cluster_sizes.size:,}</span>
        </div>
        
        <div style="width: 130px;">
             <span style="color: #586069; font-size: 0.9em; text-transform: uppercase;">Smallest</span><br>
             <span style="font-size: 1.4em; font-weight: bold; color: #24292e;">{cluster_sizes.min():,}</span>
        </div>
             
        <div style="width: 130px;">
             <span style="color: #586069; font-size: 0.9em; text-transform: uppercase;">Largest</span><br>
             <span style="font-size: 1.4em; font-weight: bold; color: #24292e;">{cluster_sizes.max():,}</span>
        </div>
             
        <div style="width: 130px;">
             <span style="color: #586069; font-size: 0.9em; text-transform: uppercase;">Mean</span><br>
             <span style="font-size: 1.4em; font-weight: bold; color: #24292e;">{cluster_sizes.mean():,.2f}</span>
        </div>
             
        <div style="width: 130px;">
             <span style="color: #586069; font-size: 0.9em; text-transform: uppercase;">Median</span><br>
             <span style="font-size: 1.4em; font-weight: bold; color: #24292e;">{np.median(cluster_sizes):,.2f}</span>
        </div>
        
    </div>
    """
    display(HTML(stats_html))
    return True

## Core plotter function

In [ ]:
def plot_cluster_distribution(min_size):
    """Filters data, triggers stats display, and renders the boxplot."""
    # Filter based on the slider value
    cluster_sizes = all_cluster_sizes[all_cluster_sizes >= min_size]

    # Display stats and check if data exists
    has_data = display_cluster_stats(cluster_sizes, min_size)
    if not has_data:
        return

    # Plot styling
    plt.style.use("seaborn-v0_8-dark-palette")
    _, ax = plt.subplots(figsize=(10, 4), dpi=150)

    box_style = {"facecolor": "#4C72B0", "color": "#4C72B0", "linewidth": 1.5}
    median_style = {"color": "#C44E52", "linewidth": 2}
    whisker_style = {"color": "#4C72B0", "linewidth": 1.5}
    capprops_style = {"color": "#4C72B0", "linewidth": 1.5}

    # Plot horizontal box plot
    ax.boxplot(
        cluster_sizes,
        orientation="horizontal",
        patch_artist=True,
        widths=0.4,
        showfliers=False,
        boxprops=box_style,
        medianprops=median_style,
        whiskerprops=whisker_style,
        capprops=capprops_style,
    )

    ax.set_xlim(left=0.5)
    ax.set_xlabel(f"Cluster size (\u2265 {min_size} members)")
    ax.set_yticks([])
    ax.set_ylabel("")
    ax.grid(axis="x", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

## Setup UI and run

Run the cell below to launch the interactive visualization.

Move the slider to explore the distribution until you find a cutoff that retains your desired data while filtering out noise. You can then use this value as the `min_cluster_size` argument in the main pipeline.

In [ ]:
if 'all_cluster_sizes' not in globals():
    print("Please load a cluster.tsv file in the filechooser above first!")
else:
    # Generate ~150 logarithmically spaced steps between 1 and the max cluster size
    max_val = int(all_cluster_sizes.max())
    slider_options = np.unique(np.geomspace(1, max_val, num=150).astype(int))

    # Find the option closest to 5 to use as starting point
    default_val = min(slider_options, key=lambda x: abs(x - 5))

    # Create slider widget
    min_size_slider = widgets.SelectionSlider(
        options=list(slider_options),
        value=default_val,
        description='Min size:',
        continuous_update=False,  
        layout=widgets.Layout(width='500px')
    )

    # Render interaction
    widgets.interact(plot_cluster_distribution, min_size=min_size_slider)